# [Hybrid RAG Made Easy: Step-by-Step with LangChain, FAISS, Azure OpenAI, LLMGraph Transformer, and NetworkX Graphs](https://pub.towardsai.net/hybrid-rag-made-easy-step-by-step-with-langchain-faiss-azureopenai-llmgraphtransformer-and-ef93cd50948d)

modified for use with OLLama

In [10]:
import os
from langchain_experimental.graph_transformers import LLMGraphTransformer
import networkx as nx
from langchain.chains import GraphQAChain
from langchain_core.documents import Document
from langchain_community.graphs.networkx_graph import NetworkxEntityGraph
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
import json
from langchain_core.documents import Document

In [11]:
llm = ChatOllama(
   model="llama3.2:latest",
   temperature=0,
   # other params...
)

embeddings = OllamaEmbeddings(model="llama3.2:latest")

In [12]:


with open("./data/training_modeling_papers.json", "r") as f:
    data = json.load(f)

training_documents = []

for row in data:
    training_documents.append(Document(page_content=row["abstract"]))

f"Papers loaded: {len(training_documents)}"

'Papers loaded: 46'

In [16]:
# Defining a generic vector based RAG function
def generic_rag(docs,index_filename):


    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap=100
    )

    texts = text_splitter.split_documents(docs)
    
    # Check if the index already exists
    if os.path.exists(index_filename):
        # Load the existing FAISS index
        docsearch = FAISS.read_index(index_filename)
    else:
        # Create a FAISS index from the text documents
        docsearch = FAISS.from_documents(texts, embeddings)
        
        # Save the FAISS index to disk
        FAISS.save_local(docsearch, index_filename)

    retriever = docsearch.as_retriever(search_type="similarity_score_threshold",search_kwargs={'k':10,'score_threshold':.1})
    qa = RetrievalQA.from_chain_type.invoke(llm=llm,chain_type="stuff",retriever=retriever)

    return qa

In [17]:
gr = generic_rag(training_documents,"index.faiss")

AttributeError: type object 'FAISS' has no attribute 'read_index'

In [15]:
gr("what is the reproduction number")

/var/folders/_8/hq_dqd_j32g040s82w869_f00000gp/T/ipykernel_24334/1707511338.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  gr("what is the reproduction number")
No relevant docs were retrieved using the relevance score threshold 0.3


{'query': 'what is the reproduction number',
 'result': "The reproduction number, also known as the basic reproduction number (R0), is a mathematical concept used in epidemiology to estimate the average number of secondary cases generated by a single infected individual in a population.\n\nIn other words, it's a measure of how easily an infectious disease spreads from one person to another. A high R0 value indicates that the disease is highly contagious and can spread quickly through a population, while a low R0 value suggests that the disease is less contagious and may be controlled more easily.\n\nThe reproduction number takes into account various factors such as:\n\n* The infectivity of the disease\n* The duration of infectiousness\n* The likelihood of transmission between individuals\n* The effectiveness of public health interventions\n\nR0 can vary depending on the specific disease, population characteristics, and other factors. It's often used to predict the potential spread of a